# 🔧 Notebook 03a — Feature Engineering

**Purpose:** Transform raw cleaned columns into model-ready numeric features. Every decision is explained and justified. The feature matrix is saved as parquet for notebook 04.

**Key principle:** All encoders (target encoder, OHE) are *fit on training data only* and applied to both train, validation, and test. This prevents target leakage through the feature engineering pipeline.

> ⚠️ **Run 03b_tfidf_features.ipynb first** to generate `tfidf_word_list.json` before running this notebook.

## Setup

In [11]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

OUTPUTS_PATH = "data"   # reads from notebook 02; tfidf_word_list.json from 03b
FIGURES_PATH = os.path.join(OUTPUTS_PATH, "figures")
os.makedirs(FIGURES_PATH, exist_ok=True)

# Load cleaned splits from Notebook 01
train_df = pd.read_parquet(os.path.join(OUTPUTS_PATH, "train_df.parquet"))
val_df   = pd.read_parquet(os.path.join(OUTPUTS_PATH, "val_df.parquet"))
test_df  = pd.read_parquet(os.path.join(OUTPUTS_PATH, "test_df.parquet"))

print(f"Train : {train_df.shape}")
print(f"Val   : {val_df.shape}")
print(f"Test  : {test_df.shape}")
print(f"\nColumns available: {train_df.columns.tolist()}")

Train : (102887, 16)
Val   : (25721, 16)
Test  : (32153, 16)

Columns available: ['blurb', 'country', 'created_at', 'currency', 'deadline', 'disable_communication', 'goal', 'id', 'launched_at', 'name', 'state', 'video', 'success', 'cat_name', 'cat_parent_name', 'loc_country']


## Feature 1 — `log_goal`

`goal` has extreme right skew — the maximum is in the hundreds of millions. A log transform compresses this to a manageable range, brings the distribution closer to normal, and prevents extreme outliers from dominating tree splits or linear model coefficients.

`log_goal = log(1 + goal)` — the +1 handles the edge case of goal = 0.

In [12]:
for df in [train_df, val_df, test_df]:
    goal = pd.to_numeric(df["goal"], errors="coerce").fillna(0).clip(lower=0)
    df["log_goal"] = np.log1p(goal)

print("log_goal stats (train):")
print(train_df["log_goal"].describe())
print(f"\nOriginal goal range: 0 to {pd.to_numeric(train_df['goal'], errors='coerce').max():,.0f}")
print(f"log_goal range    : {train_df['log_goal'].min():.2f} to {train_df['log_goal'].max():.2f}")

log_goal stats (train):
count    102887.000000
mean          8.527583
std           1.795468
min           0.693147
25%           7.378384
50%           8.517393
75%           9.615872
max          18.420681
Name: log_goal, dtype: float64

Original goal range: 0 to 100,000,000
log_goal range    : 0.69 to 18.42


## Feature 2 — `duration_days`

Campaign length = (deadline - launched_at) / 86,400 seconds per day. Kickstarter's maximum allowed campaign duration is 90 days, so we cap there. Shorter campaigns historically succeed more — this feature is one of the strongest predictors in the dataset.

In [13]:
for df in [train_df, val_df, test_df]:
    launched = pd.to_numeric(df["launched_at"], errors="coerce")
    deadline = pd.to_numeric(df["deadline"], errors="coerce")
    df["duration_days"] = ((deadline - launched) / 86400).clip(0, 90).astype(float)

print("duration_days stats (train):")
print(train_df["duration_days"].describe())
print(f"\nCampaigns exactly at 30 days: {(train_df['duration_days'].round() == 30).sum():,}")
print(f"Campaigns exactly at 60 days: {(train_df['duration_days'].round() == 60).sum():,}")

duration_days stats (train):
count    102887.000000
mean         33.682304
std          12.523563
min           1.000000
25%          29.958333
50%          30.000000
75%          36.653252
max          90.000000
Name: duration_days, dtype: float64

Campaigns exactly at 30 days: 43,432
Campaigns exactly at 60 days: 11,658


## Feature 3 — `prep_days`

Days between account creation and campaign launch. A creator who spends 3 months preparing their campaign likely puts in more effort than one who launches the same day they signed up. We winsorise at the 99th percentile because some creators created accounts years before Kickstarter even had their category — these are long-tail outliers unrelated to campaign effort.

In [14]:
for df in [train_df, val_df, test_df]:
    created  = pd.to_numeric(df.get("created_at", pd.Series(np.nan, index=df.index)), errors="coerce")
    launched = pd.to_numeric(df["launched_at"], errors="coerce")
    raw = ((launched - created) / 86400).clip(lower=0)
    cap = train_df.apply(lambda r: ((pd.to_numeric(r.get("launched_at",0), errors="coerce") - 
                                      pd.to_numeric(r.get("created_at",0), errors="coerce")) / 86400), 
                          axis=1).clip(lower=0).quantile(0.99)
    df["prep_days"] = raw.clip(upper=cap).astype(float)

print("prep_days stats (train, after winsorisation):")
print(train_df["prep_days"].describe())

prep_days stats (train, after winsorisation):
count    102887.000000
mean         45.481325
std         103.648363
min           0.003368
25%           2.996453
50%          11.854097
75%          37.223883
max         714.031742
Name: prep_days, dtype: float64


## Feature 4 — `has_video` (binary)

A missing `video` column means the creator did not attach a video — this is a deliberate choice, not missing data. We encode it as a binary feature: 1 = video present, 0 = no video. From the EDA, campaigns with videos succeed at a significantly higher rate.

In [15]:
for df in [train_df, val_df, test_df]:
    if "video" in df.columns:
        df["has_video"] = df["video"].notna().astype(int)
    else:
        df["has_video"] = 0

print("has_video distribution (train):")
print(train_df["has_video"].value_counts())
print(f"\nSuccess rate with video   : {train_df[train_df['has_video']==1]['success'].mean()*100:.1f}%")
print(f"Success rate without video: {train_df[train_df['has_video']==0]['success'].mean()*100:.1f}%")

has_video distribution (train):
has_video
1    68309
0    34578
Name: count, dtype: int64

Success rate with video   : 65.0%
Success rate without video: 40.9%


## Features 5–8 — Text Features

The `blurb` (short description) and `name` contain signal about creator effort and communication quality. We extract simple length-based features — a proper NLP approach would use TF-IDF or embeddings, but length alone captures a meaningful portion of the variance.

- **blurb_length**: character count — a longer blurb usually means more explanation
- **name_length**: character count of the title
- **blurb_word_count**: word count (different from char count — captures avg word length)
- **name_number**: 1 if the campaign title contains any digit — campaigns like "Volume 2" or "Season 3" signal an established creator with a prior audience

In [16]:
for df in [train_df, val_df, test_df]:
    df["blurb_length"]     = df["blurb"].fillna("").str.len().astype(int)
    df["name_length"]      = df["name"].fillna("").str.len().astype(int)
    df["blurb_word_count"] = df["blurb"].fillna("").str.split().str.len().fillna(0).astype(int)
    df["name_number"]      = df["name"].fillna("").str.contains(r"\d", regex=True).astype(int)

print("Text feature stats (train):")
print(train_df[["blurb_length","name_length","blurb_word_count","name_number"]].describe().T)
print(f"\nSuccess rate — name has number    : {train_df[train_df['name_number']==1]['success'].mean()*100:.1f}%")
print(f"Success rate — name no number     : {train_df[train_df['name_number']==0]['success'].mean()*100:.1f}%")

Text feature stats (train):
                     count        mean        std  min   25%    50%    75%  \
blurb_length      102887.0  105.409867  31.464558  0.0  86.0  118.0  131.0   
name_length       102887.0   34.362641  15.521759  1.0  21.0   34.0   48.0   
blurb_word_count  102887.0   17.567321   5.709214  0.0  14.0   19.0   22.0   
name_number       102887.0    0.126576   0.332499  0.0   0.0    0.0    0.0   

                    max  
blurb_length      141.0  
name_length        61.0  
blurb_word_count   43.0  
name_number         1.0  

Success rate — name has number    : 68.1%
Success rate — name no number     : 55.3%


## Features 9–13 — Campaign Metadata Features

- **goal_is_round**: Round goals ($10,000) may signal less precise budgeting. A creator who asks for exactly $8,750 has probably costed their project carefully.
- **is_usd**: USD campaigns dominate the platform and may have different dynamics (larger audience, no currency friction).
- **launched_month** (1–12): Seasonal effects — Q4 pre-holiday launches may attract more backers.
- **launched_dayofweek** (0=Mon, 6=Sun): Tuesday/Wednesday launches have historically performed best.
- **staff_pick**: Kickstarter's editorial recommendation flag. Observable at launch time and carries genuine predictive signal. Note: using it raises a mild circularity concern if the model is intended for Kickstarter's own use, but it is retained as a feature given its signal strength.

**Note — `launched_year` excluded:** The 3-month rolling success rate shows a clear upward trend over time, driven by platform maturity and backer base growth — not by individual project characteristics. Including `launched_year` would cause the model to learn this temporal trend rather than actionable project features, and would extrapolate poorly to future years beyond the training range.

In [17]:
for df in [train_df, val_df, test_df]:
    goal = pd.to_numeric(df["goal"], errors="coerce").fillna(0)
    df["goal_is_round"] = ((goal % 1000) == 0).astype(int)

    if "currency" in df.columns:
        df["is_usd"] = (df["currency"] == "USD").astype(int)
    else:
        df["is_usd"] = 0

    launched_dt = pd.to_datetime(pd.to_numeric(df["launched_at"], errors="coerce"), unit="s", utc=True)
    df["launched_month"]     = launched_dt.dt.month.astype(float)
    df["launched_dayofweek"] = launched_dt.dt.dayofweek.astype(float)
    # launched_year is intentionally excluded: the rolling success rate shows an upward
    # temporal trend driven by platform maturity, not project characteristics.
    # Including it would cause the model to learn year-as-proxy rather than actionable features,
    # and would extrapolate poorly to future years outside the training range.

print("New metadata features (train):")
print(f"\nSuccess rate — round goal  : {train_df[train_df['goal_is_round']==1]['success'].mean()*100:.1f}%")
print(f"Success rate — non-round   : {train_df[train_df['goal_is_round']==0]['success'].mean()*100:.1f}%")

New metadata features (train):

Success rate — round goal  : 51.6%
Success rate — non-round   : 64.9%


## Feature 14 — `goal_per_day`

How much money must be raised per day to hit the goal. A campaign asking for $50,000 over 5 days ($10,000/day) is a very different proposition from one asking for $50,000 over 60 days ($833/day).

In [18]:
for df in [train_df, val_df, test_df]:
    goal = pd.to_numeric(df["goal"], errors="coerce").fillna(0)
    dur  = df["duration_days"].replace(0, np.nan)
    df["goal_per_day"] = (goal / dur).fillna(0).clip(upper=goal.quantile(0.99)).astype(float)

print("goal_per_day stats (train):")
print(train_df["goal_per_day"].describe())

goal_per_day stats (train):
count    102887.000000
mean       1225.937318
std       12902.797413
min           0.016655
25%          53.259362
50%         166.666667
75%         499.870277
max      500000.000000
Name: goal_per_day, dtype: float64


## Feature 15 — `log_goal_vs_cat_median`

A $100,000 goal means something very different in the Tabletop Games category (where the median goal is ~$5,000) versus Technology (where campaigns routinely ask for $50,000+). `log_goal` captures the absolute size of the ask, but not its **relative ambition within the category**.

This feature computes:
```
goal_vs_cat_median = goal / median_goal_in_cat_name
log_goal_vs_cat_median = log(1 + goal_vs_cat_median)
```

A value > 1 means the campaign is asking for more than the typical project in its category — a harder fundraising task.

**Note:** The category medians are computed on **training data only** and applied to both splits, consistent with the target encoder approach.

In [19]:
# Compute category median goal on training data only
goal_train = pd.to_numeric(train_df["goal"], errors="coerce").fillna(0)
cat_median_goal = goal_train.groupby(train_df["cat_name"]).median()
global_median_goal = goal_train.median()

for df in [train_df, val_df, test_df]:
    goal = pd.to_numeric(df["goal"], errors="coerce").fillna(0)
    cat_med = df["cat_name"].map(cat_median_goal).fillna(global_median_goal)
    ratio = goal / cat_med.replace(0, global_median_goal)
    df["log_goal_vs_cat_median"] = np.log1p(ratio)

print("log_goal_vs_cat_median stats (train):")
print(train_df["log_goal_vs_cat_median"].describe())
print(f"\nTop 10 categories by median goal (training data):")
print(cat_median_goal.sort_values(ascending=False).head(10).apply(lambda x: f"${x:,.0f}"))

log_goal_vs_cat_median stats (train):
count    102887.000000
mean          0.939609
std           0.902541
min           0.000040
25%           0.307750
50%           0.693147
75%           1.252763
max          10.491302
Name: log_goal_vs_cat_median, dtype: float64

Top 10 categories by median goal (training data):
cat_name
Design               $154,000
Publishing            $50,000
Flight                $25,000
Wearables             $25,000
Movie Theaters        $25,000
Camera Equipment      $23,195
Sound                 $20,000
Architecture          $20,000
Restaurants           $20,000
Fabrication Tools     $20,000
Name: goal, dtype: object


## Feature 16 — TF-IDF Binary Word Features

Explored in `03a_tfidf_features.ipynb`. For each term in the shortlist, a binary feature is created:  `1` = the term appears in `blurb` or `name`, `0` = it does not.

**Selection methodology (from 03a):**
1. TF-IDF fit on training data only (`min_df=50`, `max_df=0.8`, `ngram_range=(1,2)`)
2. For each term: compute success rate among campaigns containing it, and compare to the **category-weighted average success rate** of those same campaigns
3. `diff = word_success_rate − cat_avg_success_rate`
4. Keep terms with `|diff| ≥ 10pt`, `p < 0.01`, `n ≥ 200`
5. Exclude year tokens and non-English tokens (German, Spanish, Italian) — these reflect language rather than content
6. Remove near-duplicate terms with pairwise correlation ≥ 0.8 (keep the term with higher `|diff|`)

This approach ensures selected words carry signal **beyond what `cat_name_encoded` already captures**. Geographic terms (e.g. `seattle`, `brooklyn`) are retained: unlike the metadata-based `is_california`, these are text features discovered through a systematic filter and represent a creator's deliberate choice to appeal to a local community.

In [20]:
# Load word list from 03a (derived from training data only)
with open(os.path.join(OUTPUTS_PATH, "tfidf_word_list.json")) as f:
    tfidf_word_list = json.load(f)

blurb_terms = [w["term"] for w in tfidf_word_list["blurb"]]
name_terms  = [w["term"] for w in tfidf_word_list["name"]]

for df in [train_df, val_df, test_df]:
    blurb_text = df["blurb"].fillna("").str.lower()
    name_text  = df["name"].fillna("").str.lower()
    for term in blurb_terms:
        col = "blurb_has_" + term.replace(" ", "_")
        df[col] = blurb_text.str.contains(term, regex=False).astype(int)
    for term in name_terms:
        col = "name_has_" + term.replace(" ", "_")
        df[col] = name_text.str.contains(term, regex=False).astype(int)

blurb_tfidf_cols = ["blurb_has_" + t.replace(" ", "_") for t in blurb_terms]
name_tfidf_cols  = ["name_has_"  + t.replace(" ", "_") for t in name_terms]

print(f"blurb TF-IDF binary features : {len(blurb_tfidf_cols)}")
print(f"name  TF-IDF binary features : {len(name_tfidf_cols)}")
print(f"Total TF-IDF features added  : {len(blurb_tfidf_cols) + len(name_tfidf_cols)}")
print(f"\nSample blurb features: {blurb_tfidf_cols[:5]}")
print(f"Sample name  features: {name_tfidf_cols[:5]}")

blurb TF-IDF binary features : 68
name  TF-IDF binary features : 31
Total TF-IDF features added  : 99

Sample blurb features: ['blurb_has_queer', 'blurb_has_fringe', 'blurb_has_edinburgh', 'blurb_has_pi', 'blurb_has_installation']
Sample name  features: ['name_has_fringe', 'name_has_issue', 'name_has_brewing', 'name_has_fine', 'name_has_science']


## Feature 17 — Target Encoding for `cat_name`

### Why target encoding instead of OHE

`cat_name` has 160+ distinct subcategories. One-hot encoding would produce 160 sparse binary columns, leading to (1) slower training, (2) overfitting on rare categories, and (3) unnecessary feature-space expansion.

**Target encoding** replaces each category with a single numeric value — its mean success rate. This directly communicates the "likelihood of success" ordering between categories to the model.

---

### Smoothed target encoding

A plain mean success rate is unstable for categories with few samples (e.g., n=3 with 3 successes → 100% rate). We apply **smoothing** to regularize these estimates:

```
encoded = (n × cat_mean + λ × global_mean) / (n + λ)
```

| Symbol | Meaning |
|--------|---------|
| `n` | Number of samples in the category |
| `cat_mean` | Category-level mean success rate |
| `global_mean` | Overall mean success rate (≈ 60.8%) |
| `λ` | Smoothing strength (= 10) |

**What λ = 10 means in practice:**
- n = 10: blends `cat_mean` and `global_mean` equally 50:50 (heavy regularization)
- n = 50: weights `cat_mean` at ~83% (moderate trust)
- n = 200: weights `cat_mean` at ~95% (close to raw estimate)

Most subcategories in the training data have hundreds of samples, so λ=10 applies meaningful regularization only to rare categories while leaving well-represented categories largely unaffected.

In [21]:
def fit_target_encoder(train_df, col, target="success", smoothing=10.0):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(["mean", "count"])
    encoder = {}
    for cat, row in stats.iterrows():
        n = row["count"]
        encoded = (n * row["mean"] + smoothing * global_mean) / (n + smoothing)
        encoder[cat] = encoded
    encoder["__global_mean__"] = global_mean
    return encoder

def apply_target_encoder(df, col, encoder, new_col):
    global_mean = encoder.get("__global_mean__", 0.5)
    df[new_col] = df[col].map(encoder).fillna(global_mean)
    return df

# Fit on training data ONLY
cat_name_encoder = fit_target_encoder(train_df, "cat_name")

# Apply to all splits (val and test use training statistics)
train_df = apply_target_encoder(train_df, "cat_name", cat_name_encoder, "cat_name_encoded")
val_df   = apply_target_encoder(val_df,   "cat_name", cat_name_encoder, "cat_name_encoded")
test_df  = apply_target_encoder(test_df,  "cat_name", cat_name_encoder, "cat_name_encoded")

print("cat_name_encoded (target-encoded) stats (train):")
print(train_df["cat_name_encoded"].describe())
print(f"\nTop 10 subcategories by encoded value (success rate):")
top = sorted([(k,v) for k,v in cat_name_encoder.items() if k != "__global_mean__"], key=lambda x: -x[1])[:10]
for k,v in top:
    print(f"  {k:<35} : {v*100:.1f}%")

cat_name_encoded (target-encoded) stats (train):
count    102887.000000
mean          0.569653
std           0.278128
min           0.148772
25%           0.335253
50%           0.519784
75%           0.837551
max           0.995933
Name: cat_name_encoded, dtype: float64

Top 10 subcategories by encoded value (success rate):
  Indie Rock                          : 99.6%
  Country & Folk                      : 99.5%
  Rock                                : 99.5%
  Painting                            : 99.5%
  Art Books                           : 99.5%
  Drama                               : 99.4%
  Documentary                         : 99.3%
  Narrative Film                      : 99.3%
  Crafts                              : 99.3%
  Apparel                             : 99.3%


## Feature 18 — Target Encoding for `cat_parent_name`

There are only 15 parent categories, so one-hot encoding is technically feasible. However, for consistency with `cat_name_encoded` and to give the model ordinal information directly, we apply the same smoothed target encoding approach.

This replaces 15 sparse OHE binary columns with a single numeric column representing each parent category's mean success rate — the same logic as Feature 17 but at the coarser parent-category level.

The two features capture different granularities of the same signal:
- `cat_name_encoded`: subcategory-level success rate (fine-grained)
- `cat_parent_encoded`: parent-category-level success rate (coarse-grained)

### Investigation: "Unknown" parent category — genuine signal or data artefact?

**Finding:** `cat_parent_name == 'Unknown'` contains **4,210 campaigns** (94% success rate in the full dataset; 97% in train).

**Sub-categories within "Unknown":** Dance, Theater, Crafts, Photography, Journalism, Food, Art — all legitimate arts/creative categories.

**Year distribution:** "Unknown" campaigns appear across all years (2009–2026), not concentrated in early years.

**Year-by-year comparison:** "Unknown" campaigns consistently outperform the platform average in every year — including 2014–2017 when the overall success rate dropped to ~47–55% but Unknown remained at 100%.

**Conclusion: the high success rate is a genuine signal, but one that is already captured elsewhere.**

The "Unknown" parent label is a **legacy data artefact** from Kickstarter's old category system: campaigns created before or during a category hierarchy migration were assigned to sub-categories (Dance, Theater, etc.) without a parent being recorded. These sub-categories happen to be high-success arts categories.

- The 94% rate reflects the underlying sub-category composition, not something special about being uncategorised.
- `cat_name_encoded` already captures this signal directly (Dance, Theater, etc. all have high encoded values).
- `cat_parent_encoded` for "Unknown" is therefore a **redundant proxy** — it adds no information beyond what `cat_name_encoded` provides, and its inflated value could confuse linear models.

**Decision:** Keep the feature as-is (the target encoder with smoothing will partially dampen the extreme value), but note that its predictive weight in linear models should be interpreted with caution. If model coefficients for `cat_parent_encoded` appear unexpectedly large, consider replacing "Unknown" values with the global mean before encoding.

In [22]:
# Fit on training data ONLY
cat_parent_encoder = fit_target_encoder(train_df, "cat_parent_name")

# Apply to all splits (val and test use training statistics)
train_df = apply_target_encoder(train_df, "cat_parent_name", cat_parent_encoder, "cat_parent_encoded")
val_df   = apply_target_encoder(val_df,   "cat_parent_name", cat_parent_encoder, "cat_parent_encoded")
test_df  = apply_target_encoder(test_df,  "cat_parent_name", cat_parent_encoder, "cat_parent_encoded")

print("cat_parent_encoded (target-encoded) stats (train):")
print(train_df["cat_parent_encoded"].describe())
print(f"\nAll parent categories by encoded value (success rate):")
top = sorted([(k,v) for k,v in cat_parent_encoder.items() if k != "__global_mean__"], key=lambda x: -x[1])
for k,v in top:
    print(f"  {k:<20} : {v*100:.1f}%")

cat_parent_encoded (target-encoded) stats (train):
count    102887.000000
mean          0.569241
std           0.133278
min           0.256157
25%           0.507905
50%           0.639964
75%           0.661236
max           0.949460
Name: cat_parent_encoded, dtype: float64

All parent categories by encoded value (success rate):
  Unknown              : 94.9%
  Comics               : 70.1%
  Film & Video         : 67.2%
  Music                : 66.1%
  Art                  : 65.4%
  Publishing           : 64.0%
  Theater              : 61.5%
  Dance                : 60.6%
  Technology           : 53.4%
  Photography          : 51.8%
  Design               : 51.8%
  Fashion              : 50.8%
  Games                : 42.0%
  Food                 : 39.2%
  Crafts               : 31.2%
  Journalism           : 25.6%


## Feature 19 — One-Hot Encode `country`

Group countries with fewer than 500 campaigns into 'Other' to avoid sparse, unreliable columns. Fit on training data only.

In [23]:
# Group rare countries
country_counts = train_df["country"].value_counts()
keep_countries = country_counts[country_counts >= 500].index.tolist()
print(f"Countries with ≥500 campaigns: {keep_countries}")
print(f"Countries grouped into 'Other': {country_counts[country_counts < 500].index.tolist()}")

for df in [train_df, val_df, test_df]:
    df["country_grouped"] = df["country"].where(df["country"].isin(keep_countries), other="Other")
    country_dummies = pd.get_dummies(df["country_grouped"], prefix="country").astype(int)
    for col in country_dummies.columns:
        df[col] = country_dummies[col]

country_cols = [c for c in train_df.columns if c.startswith("country_") and train_df[c].dtype != object]
print(f"\nAdded {len(country_cols)} country OHE columns: {country_cols}")

Countries with ≥500 campaigns: ['US', 'GB', 'CA', 'DE', 'AU', 'MX', 'FR', 'IT', 'ES', 'SE', 'NL', 'HK', 'DK']
Countries grouped into 'Other': ['JP', 'SG', 'NZ', 'BE', 'CH', 'IE', 'AT', 'NO', 'PL', 'GR', 'LU', 'SI']

Added 14 country OHE columns: ['country_AU', 'country_CA', 'country_DE', 'country_DK', 'country_ES', 'country_FR', 'country_GB', 'country_HK', 'country_IT', 'country_MX', 'country_NL', 'country_Other', 'country_SE', 'country_US']


## Feature Summary Table

In [24]:
engineered_features = [
    # --- Numeric features ---
    ("log_goal",                "numeric",  "Log(1+goal): compresses right skew of funding goal"),
    ("duration_days",           "numeric",  "Campaign duration in days (capped at 90)"),
    ("prep_days",               "numeric",  "Days from account creation to launch (winsorised at 99th pct)"),
    ("blurb_length",            "numeric",  "Character count of campaign short description"),
    ("name_length",             "numeric",  "Character count of campaign title"),
    ("blurb_word_count",        "numeric",  "Word count of campaign short description"),
    ("launched_month",          "numeric",  "Month of launch (1=Jan, 12=Dec)"),
    ("launched_dayofweek",      "numeric",  "Day of week at launch (0=Mon, 6=Sun)"),
    ("goal_per_day",            "numeric",  "goal / duration_days: daily fundraising target"),
    ("log_goal_vs_cat_median",  "numeric",  "Log(1 + goal / category_median_goal): relative ambition within category"),
    ("cat_name_encoded",        "numeric",  "Smoothed target encoding of subcategory (train success rate, λ=10)"),
    ("cat_parent_encoded",      "numeric",  "Smoothed target encoding of parent category (train success rate, λ=10)"),
    # --- Binary features ---
    ("has_video",               "binary",   "1 = video attached to campaign page, 0 = no video"),
    ("name_number",             "binary",   "1 = campaign title contains a digit (0–9)"),
    ("goal_is_round",           "binary",   "1 = funding goal is divisible by 1,000"),
    ("is_usd",                  "binary",   "1 = campaign currency is USD"),
    ("staff_pick",              "binary",   "1 = campaign flagged as Kickstarter editorial pick"),
]

# Country OHE features (dynamic — depends on training data)
country_cols = sorted([c for c in train_df.columns if c.startswith("country_") and train_df[c].dtype != object])
country_rows = [(col, "binary", f"1 = campaign from {col.replace('country_', '')} (one-hot encoded)")
                for col in country_cols]

# TF-IDF binary features (dynamic — loaded from tfidf_word_list.json)
blurb_tfidf_cols = sorted([c for c in train_df.columns if c.startswith("blurb_has_")])
name_tfidf_cols  = sorted([c for c in train_df.columns if c.startswith("name_has_")])
blurb_rows = [(col, "binary", f"1 = blurb contains '{col.replace('blurb_has_', '').replace('_', ' ')}'")
              for col in blurb_tfidf_cols]
name_rows  = [(col, "binary", f"1 = title contains '{col.replace('name_has_', '').replace('_', ' ')}'")
              for col in name_tfidf_cols]

all_rows = engineered_features + country_rows + blurb_rows + name_rows
summary = pd.DataFrame(all_rows, columns=["feature", "type", "description"])
summary["missing_rate_train"] = summary["feature"].apply(
    lambda f: f"{train_df[f].isnull().mean()*100:.2f}%" if f in train_df.columns else "—"
)

print(f"Total features: {len(all_rows)}")
print(f"  Numeric/target-encoded : {len([r for r in engineered_features if r[1] == 'numeric'])}")
print(f"  Binary (metadata)      : {len([r for r in engineered_features if r[1] == 'binary'])}")
print(f"  Country OHE            : {len(country_rows)}")
print(f"  TF-IDF blurb           : {len(blurb_rows)}")
print(f"  TF-IDF name            : {len(name_rows)}")
print()
print(summary.to_string(index=False))

os.makedirs(os.path.join(OUTPUTS_PATH, "results"), exist_ok=True)
summary.to_csv(os.path.join(OUTPUTS_PATH, "results", "feature_summary.csv"), index=False)
print(f"\nSaved feature_summary.csv")

Total features: 130
  Numeric/target-encoded : 12
  Binary (metadata)      : 5
  Country OHE            : 14
  TF-IDF blurb           : 68
  TF-IDF name            : 31

                  feature    type                                                             description missing_rate_train
                 log_goal numeric                      Log(1+goal): compresses right skew of funding goal              0.00%
            duration_days numeric                                Campaign duration in days (capped at 90)              0.00%
                prep_days numeric           Days from account creation to launch (winsorised at 99th pct)              0.00%
             blurb_length numeric                           Character count of campaign short description              0.00%
              name_length numeric                                       Character count of campaign title              0.00%
         blurb_word_count numeric                                Word count of c

## Build Final Feature Matrix and Save

The feature matrices are saved as-is (unscaled). Tree-based models (Random Forest, XGBoost, Gradient Boosting) do not require feature scaling.

**Note for Logistic Regression:** numeric features have different scales (e.g. `log_goal` ≈ 0–20, `duration_days` ≈ 0–90, `blurb_length` ≈ 0–500). When training Logistic Regression, apply `StandardScaler` inside a pipeline so that scaling is fit on training data only and applied consistently to validation and test:

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression())
])
```

In [25]:
BASE_FEATURES = [
    "log_goal", "duration_days", "prep_days", "has_video",
    "blurb_length", "name_length", "blurb_word_count", "name_number",
    "goal_is_round", "is_usd", "launched_month", "launched_dayofweek",
    "goal_per_day", "log_goal_vs_cat_median",
    "cat_name_encoded", "cat_parent_encoded",
]
OHE_FEATURES   = [c for c in train_df.columns if c.startswith("country_") and train_df[c].dtype != object]
TFIDF_FEATURES = blurb_tfidf_cols + name_tfidf_cols

FEATURE_COLS = [f for f in BASE_FEATURES + OHE_FEATURES + TFIDF_FEATURES if f in train_df.columns]
print(f"Total features: {len(FEATURE_COLS)}")

Total features: 129


In [26]:
import json as _json
with open(os.path.join(OUTPUTS_PATH, 'feature_cols.json'), 'w') as _f:
    _json.dump(FEATURE_COLS, _f)
print(f'Saved feature_cols.json — {len(FEATURE_COLS)} features')

Saved feature_cols.json — 129 features


In [27]:
# Save feature matrices and targets for the modelling notebook
X_train = train_df[FEATURE_COLS].fillna(0)
X_val   = val_df[FEATURE_COLS].fillna(0)
X_test  = test_df[FEATURE_COLS].fillna(0)
y_train = train_df['success']
y_val   = val_df['success']
y_test  = test_df['success']

X_train.to_parquet(os.path.join(OUTPUTS_PATH, 'X_train.parquet'), index=False)
X_val.to_parquet(os.path.join(OUTPUTS_PATH,   'X_val.parquet'),   index=False)
X_test.to_parquet(os.path.join(OUTPUTS_PATH,  'X_test.parquet'),  index=False)
y_train.to_frame().to_parquet(os.path.join(OUTPUTS_PATH, 'y_train.parquet'), index=False)
y_val.to_frame().to_parquet(os.path.join(OUTPUTS_PATH,   'y_val.parquet'),   index=False)
y_test.to_frame().to_parquet(os.path.join(OUTPUTS_PATH,  'y_test.parquet'),  index=False)

print(f'X_train : {X_train.shape}')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')
print('Saved X_train, X_val, X_test, y_train, y_val, y_test parquets.')

X_train : (102887, 129)
X_val   : (25721, 129)
X_test  : (32153, 129)
Saved X_train, X_val, X_test, y_train, y_val, y_test parquets.
